# Ru'ya YOLOv8 Trained Model Inference Pipeline

        This notebook uses the trained `best.pt` model directly. It is cleaner than the training notebook for final demonstration because it focuses on:

        - loading the trained model,
        - processing the UAV test video,
        - drawing detections and grid-based risk zones,
        - saving a final output video,
        - exporting frame-level statistics,
        - saving potential failure-case frames for qualitative analysis.

        Use this notebook when you already have the trained model and want reliable demo/evaluation code.


In [ ]:
# Run this in Colab/Kaggle if ultralytics is not already installed.
        # In Kaggle, internet may need to be enabled.
        !pip install ultralytics opencv-python pandas matplotlib -q


In [ ]:
from pathlib import Path
        import time

        import cv2
        import numpy as np
        import pandas as pd
        import matplotlib.pyplot as plt
        from ultralytics import YOLO
        from IPython.display import Video, display


## Configuration

        The paths below work on your Windows machine. If you run this in Colab/Kaggle, upload `best.pt` and `drone_test.mp4`, then change the paths to `best.pt` and `drone_test.mp4`.


In [ ]:
# Windows paths from your current files.
        MODEL_PATH = Path(r"C:\Users\djood\Downloads\best.pt")
        ONNX_PATH = Path(r"C:\Users\djood\Downloads\best.onnx")
        INPUT_VIDEO = Path(r"C:\Users\djood\Downloads\drone_test.mp4")

        # Colab/Kaggle fallback paths if the Windows paths do not exist.
        if not MODEL_PATH.exists():
            MODEL_PATH = Path("best.pt")
        if not ONNX_PATH.exists():
            ONNX_PATH = Path("best.onnx")
        if not INPUT_VIDEO.exists():
            INPUT_VIDEO = Path("drone_test.mp4")

        OUTPUT_VIDEO = Path("ruya_better_crowd_output.mp4")
        STATS_CSV = Path("ruya_frame_statistics.csv")
        FAILURE_DIR = Path("failure_case_frames")
        FAILURE_DIR.mkdir(exist_ok=True)

        # Detection settings.
        # 1280 is better for tiny UAV people than 640, but slower.
        IMG_SIZE = 1280
        CONF_THRES = 0.25
        IOU_THRES = 0.50
        MAX_DET = 1000

        # Grid/risk settings. These are count thresholds per grid cell.
        GRID_ROWS = 6
        GRID_COLS = 6
        MEDIUM_RISK_COUNT = 2
        HIGH_RISK_COUNT = 4

        print("Model:", MODEL_PATH)
        print("Video:", INPUT_VIDEO)
        print("Output:", OUTPUT_VIDEO)


In [ ]:
assert MODEL_PATH.exists(), f"Model file not found: {MODEL_PATH}"
        assert INPUT_VIDEO.exists(), f"Input video not found: {INPUT_VIDEO}"

        model = YOLO(str(MODEL_PATH))
        print("Model loaded successfully.")


## Helper Functions

        The risk score is intentionally based on the **number of detected people per grid cell**. This matches the thresholds used in the project report better than using pixel-area density.


In [ ]:
def brightness_score(frame):
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            return float(gray.mean())


        def detect_people(frame):
            results = model.predict(
                frame,
                imgsz=IMG_SIZE,
                conf=CONF_THRES,
                iou=IOU_THRES,
                max_det=MAX_DET,
                verbose=False,
            )
            boxes_obj = results[0].boxes
            if boxes_obj is None or len(boxes_obj) == 0:
                return np.empty((0, 4), dtype=int), np.empty((0,), dtype=float)

            boxes = boxes_obj.xyxy.cpu().numpy().astype(int)
            confs = boxes_obj.conf.cpu().numpy()
            return boxes, confs


        def grid_counts_for_boxes(boxes, frame_shape):
            h, w = frame_shape[:2]
            cell_w = w / GRID_COLS
            cell_h = h / GRID_ROWS
            grid_counts = np.zeros((GRID_ROWS, GRID_COLS), dtype=int)

            for x1, y1, x2, y2 in boxes:
                cx = (x1 + x2) / 2
                cy = (y1 + y2) / 2
                gx = min(int(cx // cell_w), GRID_COLS - 1)
                gy = min(int(cy // cell_h), GRID_ROWS - 1)
                if gx >= 0 and gy >= 0:
                    grid_counts[gy, gx] += 1

            return grid_counts


        def risk_color_and_label(count):
            if count >= HIGH_RISK_COUNT:
                return (0, 0, 255), "HIGH"
            if count >= MEDIUM_RISK_COUNT:
                return (0, 255, 255), "MED"
            return (0, 180, 0), "LOW"


In [ ]:
def draw_overlay(frame, boxes, confs, grid_counts, fps=None):
            annotated = frame.copy()
            h, w = annotated.shape[:2]
            cell_w = w / GRID_COLS
            cell_h = h / GRID_ROWS

            # Draw person detections.
            for (x1, y1, x2, y2), conf in zip(boxes, confs):
                cv2.rectangle(annotated, (x1, y1), (x2, y2), (255, 80, 0), 2)
                cv2.putText(
                    annotated,
                    f"person {conf:.2f}",
                    (x1, max(15, y1 - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.45,
                    (255, 80, 0),
                    1,
                    cv2.LINE_AA,
                )

            # Draw risk grid.
            for row in range(GRID_ROWS):
                for col in range(GRID_COLS):
                    count = int(grid_counts[row, col])
                    color, label = risk_color_and_label(count)
                    x1 = int(col * cell_w)
                    y1 = int(row * cell_h)
                    x2 = int((col + 1) * cell_w)
                    y2 = int((row + 1) * cell_h)
                    cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
                    cv2.putText(
                        annotated,
                        f"{label}:{count}",
                        (x1 + 5, y1 + 22),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.55,
                        color,
                        2,
                        cv2.LINE_AA,
                    )

            # Top-left summary panel.
            panel_h = 92
            cv2.rectangle(annotated, (10, 10), (360, panel_h), (0, 0, 0), -1)
            cv2.putText(annotated, f"People detected: {len(boxes)}", (22, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2)
            if fps is not None:
                cv2.putText(annotated, f"Processing FPS: {fps:.2f}", (22, 72), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2)

            return annotated


## Process the Video

        This produces:

        - `ruya_better_crowd_output.mp4`
        - `ruya_frame_statistics.csv`
        - selected frames in `failure_case_frames/`


In [ ]:
cap = cv2.VideoCapture(str(INPUT_VIDEO))
        assert cap.isOpened(), f"Could not open video: {INPUT_VIDEO}"

        source_fps = cap.get(cv2.CAP_PROP_FPS) or 10
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        writer = cv2.VideoWriter(
            str(OUTPUT_VIDEO),
            cv2.VideoWriter_fourcc(*"mp4v"),
            source_fps,
            (width, height),
        )

        rows = []
        frame_index = 0
        start = time.time()

        while True:
            ok, frame = cap.read()
            if not ok:
                break

            frame_start = time.time()
            boxes, confs = detect_people(frame)
            grid_counts = grid_counts_for_boxes(boxes, frame.shape)
            current_fps = 1 / max(time.time() - frame_start, 1e-6)
            annotated = draw_overlay(frame, boxes, confs, grid_counts, fps=current_fps)
            writer.write(annotated)

            avg_conf = float(confs.mean()) if len(confs) else 0.0
            min_conf = float(confs.min()) if len(confs) else 0.0
            bright = brightness_score(frame)
            max_cell_count = int(grid_counts.max()) if grid_counts.size else 0

            # Heuristic potential failure cases for human review:
            # low light, no/low detections, or generally weak confidence.
            potential_failure = (
                bright < 75
                or len(boxes) == 0
                or avg_conf < 0.35
            )
            if potential_failure and frame_index % 20 == 0:
                cv2.imwrite(str(FAILURE_DIR / f"frame_{frame_index:05d}_potential_failure.jpg"), annotated)

            rows.append({
                "frame": frame_index,
                "people_detected": len(boxes),
                "avg_confidence": avg_conf,
                "min_confidence": min_conf,
                "brightness": bright,
                "max_grid_cell_count": max_cell_count,
                "processing_fps": current_fps,
                "potential_failure_case": potential_failure,
            })

            frame_index += 1
            if frame_index % 25 == 0:
                print(f"Processed {frame_index}/{total_frames} frames")

        cap.release()
        writer.release()

        stats = pd.DataFrame(rows)
        stats.to_csv(STATS_CSV, index=False)

        print("Done.")
        print("Frames processed:", frame_index)
        print("Average processing FPS:", frame_index / max(time.time() - start, 1e-6))
        print("Output video:", OUTPUT_VIDEO)
        print("Statistics CSV:", STATS_CSV)
        print("Potential failure frames:", FAILURE_DIR)


In [ ]:
display(Video(str(OUTPUT_VIDEO), embed=True))


## Frame-Level Summary

        These statistics help support the report discussion. They do not replace ground-truth evaluation, but they are useful for explaining practical behavior on the demo video.


In [ ]:
stats = pd.read_csv(STATS_CSV)
        display(stats.head())

        print("Average people detected per frame:", stats["people_detected"].mean())
        print("Max people detected in one frame:", stats["people_detected"].max())
        print("Average confidence:", stats["avg_confidence"].mean())
        print("Potential failure-case frames:", int(stats["potential_failure_case"].sum()))


In [ ]:
plt.figure(figsize=(12, 4))
        plt.plot(stats["frame"], stats["people_detected"], label="Detected people")
        plt.xlabel("Frame")
        plt.ylabel("Count")
        plt.title("Detected People per Frame")
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.show()

        plt.figure(figsize=(12, 4))
        plt.plot(stats["frame"], stats["avg_confidence"], label="Average confidence", color="orange")
        plt.xlabel("Frame")
        plt.ylabel("Confidence")
        plt.title("Average Detection Confidence per Frame")
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.show()


## Optional: ONNX Model Check

        Use the ONNX model mainly for deployment/CPU inference tests. For final report screenshots, the PyTorch `best.pt` model is usually simpler and more reliable inside Ultralytics.


In [ ]:
if ONNX_PATH.exists():
            onnx_model = YOLO(str(ONNX_PATH))
            print("ONNX model can be loaded:", ONNX_PATH)
        else:
            print("ONNX file not found. This is optional.")


## Suggested Report Explanation

        This improved pipeline uses the trained YOLOv8s model with higher-resolution inference (`imgsz=1280`) to better preserve small human targets in UAV imagery. The grid risk module uses count-based thresholds per grid cell, which is clearer and more consistent with the implemented warning logic. Potential failure-case frames are exported for qualitative review, especially under low-light, weak-confidence, or low-detection conditions.
